<a href="https://colab.research.google.com/github/leman-cap13/NLP_projects/blob/main/Subword_level_tokenization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Sentiment Analysis Using Hugging Face Libraries

In [ ]:
from datasets import load_dataset

imdb_dataset = load_dataset("imdb")

In [ ]:
imdb_dataset

In [ ]:
split = imdb_dataset["train"].train_test_split(train_size=0.8, seed=42)

imdb_train_set = split["train"]
imdb_valid_set = split["test"]
imdb_test_set = imdb_dataset["test"]

In [ ]:
print(imdb_train_set[1]["text"])
print(imdb_train_set[1]["label"])

#Byte pair encoding (BPE)

In [ ]:
import tokenizers


In [ ]:
bpe_model = tokenizers.models.BPE(unk_token="<unk>")
bpe_model

In [ ]:
bpe_tokenizer = tokenizers.Tokenizer(bpe_model)
bpe_tokenizer

In [ ]:
bpe_tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.Whitespace()


In [48]:
special_tokens = ["<pad>", "<unk>"] # pad , unk


In [ ]:
bpe_trainer = tokenizers.trainers.BpeTrainer(
    vocab_size=1000,
    special_tokens=special_tokens
)

In [ ]:
train_reviews = [review["text"].lower() for review in imdb_train_set]

train_reviews[0]

In [ ]:
bpe_tokenizer.train_from_iterator(train_reviews, bpe_trainer)

In [ ]:
# import tokenizers

# bpe_model = tokenizers.models.BPE(unk_token="<unk>")
# bpe_tokenizer = tokenizers.Tokenizer(bpe_model)

# bpe_tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.Whitespace()

# special_tokens = ["<pad>", "<unk>"]

# bpe_trainer = tokenizers.trainers.BpeTrainer(
#     vocab_size=1000,
#     special_tokens=special_tokens
# )

# train_reviews = [review["text"].lower() for review in imdb_train_set]

# bpe_tokenizer.train_from_iterator(train_reviews, bpe_trainer)

In [ ]:
some_review = "what an awesome movie! 😊"

bpe_encoding = bpe_tokenizer.encode(some_review)

In [ ]:
print(bpe_encoding.tokens)

In [ ]:
bpe_token_ids = bpe_encoding.ids
print(bpe_token_ids)

In [ ]:
decoded_text = bpe_tokenizer.decode(bpe_token_ids)
print(decoded_text)

In [ ]:
print(bpe_encoding.offsets)

In [ ]:
bpe_encodings = bpe_tokenizer.encode_batch(train_reviews[:3])

In [ ]:
bpe_tokenizer.enable_padding(
    pad_id=0,
    pad_token="<pad>"
)

bpe_tokenizer.enable_truncation(max_length=500)

In [ ]:
import torch

bpe_encodings = bpe_tokenizer.encode_batch(train_reviews[:3])

bpe_batch_ids = torch.tensor(
    [encoding.ids for encoding in bpe_encodings]
)

print(bpe_batch_ids)

In [ ]:
attention_mask = torch.tensor(
    [encoding.attention_mask for encoding in bpe_encodings]
)

print(attention_mask)

In [ ]:
lengths = attention_mask.sum(dim=-1)
print(lengths)

#Byte-level BPE / BBPE

In [ ]:
bpe_tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.ByteLevel()


In [ ]:
bpe_tokenizer.decoder = tokenizers.decoders.ByteLevel()


In [ ]:
special_tokens = ["<pad>", "<unk>"]

bpe_trainer = tokenizers.trainers.BpeTrainer(
    vocab_size=1000,
    special_tokens=special_tokens
)

In [ ]:
bpe_tokenizer.train_from_iterator(
    train_reviews,
    bpe_trainer
)

In [ ]:
some_review = "what an awesome movie!"

encoding = bpe_tokenizer.encode(some_review)

In [ ]:
print(encoding.tokens)
print(encoding.ids)
print(bpe_tokenizer.decode(encoding.ids))

#WordPiece

In [ ]:
wordpiece_model = tokenizers.models.WordPiece(
    unk_token="[UNK]"
)

wordpiece_tokenizer = tokenizers.Tokenizer(
    wordpiece_model
)

In [ ]:
wordpiece_tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.Whitespace()


In [ ]:
wordpiece_trainer = tokenizers.trainers.WordPieceTrainer(
    vocab_size=1000,
    special_tokens=["[PAD]", "[UNK]"]
)

In [ ]:
wordpiece_tokenizer.train_from_iterator(
    train_reviews,
    wordpiece_trainer
)

In [ ]:
some_review = "what an awesome movie! 😊"

encoding = wordpiece_tokenizer.encode(some_review)

print("TOKENS:")
print(encoding.tokens)

print("\nIDS:")
print(encoding.ids)

print("\nDECODED:")
print(wordpiece_tokenizer.decode(encoding.ids))

In [ ]:
wordpiece_tokenizer.decoder = tokenizers.decoders.WordPiece(
    prefix="##"
)

wordpiece_trainer = tokenizers.trainers.WordPieceTrainer(
    vocab_size=1000,
    special_tokens=["[PAD]", "[UNK]"],
    continuing_subword_prefix="##"
)

wordpiece_tokenizer.train_from_iterator(
    train_reviews,
    wordpiece_trainer
)


some_review = "what an awesome movie! 😊"

encoding = wordpiece_tokenizer.encode(some_review)

print("TOKENS:")
print(encoding.tokens)

print("\nIDS:")
print(encoding.ids)

print("\nDECODED:")
print(wordpiece_tokenizer.decode(encoding.ids))

#Unigram LM

In [ ]:
unigram_model = tokenizers.models.Unigram()

unigram_tokenizer = tokenizers.Tokenizer(
    unigram_model
)


In [ ]:
unigram_tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.Metaspace(
    replacement="▁",
    prepend_scheme="always"
)


In [ ]:
unigram_tokenizer.decoder = tokenizers.decoders.Metaspace(
    replacement="▁",
    prepend_scheme="always"
)

In [ ]:
unigram_trainer = tokenizers.trainers.UnigramTrainer(
    vocab_size=1000,
    special_tokens=["<pad>", "<unk>"],
    unk_token="<unk>"
)


In [ ]:
unigram_tokenizer.train_from_iterator(
    train_reviews,
    unigram_trainer
)

In [ ]:
some_review = "what an awesome movie!"

encoding = unigram_tokenizer.encode(some_review)

print("TOKENS:")
print(encoding.tokens)

print("\nIDS:")
print(encoding.ids)

print("\nDECODED:")
print(unigram_tokenizer.decode(encoding.ids))